In [170]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split  
import pandas as pd
#分块处理大文件
# chunk_iter = pd.read_csv("data/train_data.csv", chunksize=10000)
# chunks = [chunk for chunk in chunk_iter]
# train_data = pd.concat(chunks)
train_data = pd.read_csv("train/train.csv")
test_data = pd.read_csv("testaa/testaa.csv")

train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53480 entries, 0 to 53479
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                53480 non-null  int64  
 1   title             53480 non-null  int64  
 2   career            50872 non-null  float64
 3   zip_code          53480 non-null  int64  
 4   residence         53480 non-null  int64  
 5   loan              53480 non-null  int64  
 6   term              53480 non-null  int64  
 7   interest_rate     53480 non-null  float64
 8   issue_time        53480 non-null  int64  
 9   syndicated        53480 non-null  int64  
 10  installment       53480 non-null  int64  
 11  record_time       53480 non-null  int64  
 12  history_time      53480 non-null  int64  
 13  total_accounts    53480 non-null  float64
 14  balance_accounts  53480 non-null  float64
 15  balance_limit     53350 non-null  float64
 16  balance           53480 non-null  float6

In [ ]:
###   特征工程
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# 删除zip-code
train_data.drop('zip_code',axis=1, inplace=True)
test_data.drop('zip_code',axis=1, inplace=True)

# 1. 初始自动分离（基于数据类型）
numerical_fea = train_data.select_dtypes(exclude=['object']).columns.tolist()
category_fea = train_data.select_dtypes(include=['object']).columns.tolist()
numerical_fea.remove('label')  # 移除标签列

# 2. 手动调整特定字段类型
features_to_convert = ['title', 'career','residence']  # 需重分类的特征

# 将指定特征从类别型移至数值型
for feature in features_to_convert:
    if feature in category_fea:
        category_fea.remove(feature)
        numerical_fea.append(feature)

# 3. 缺失值处理
# 数值特征：中位数填充 balance_limit
train_data[numerical_fea] = train_data[numerical_fea].fillna(train_data[numerical_fea].median())
test_data[numerical_fea] = test_data[numerical_fea].fillna(test_data[numerical_fea].median())
# 类别特征：众数填充/-1填充 career
train_data[category_fea] = train_data[category_fea].fillna(-1)
test_data[category_fea] = test_data[category_fea].fillna(-1)


# 4. 类别特征编码
def level_to_num(level):
    grade_map = {'A':0, 'B':1, 'C':2, 'D':3, 'E':4}
    grade = level[0]              # A/B/C/D/E
    sub = int(level[1])           # 0-5
    return grade_map[grade] * 6 + sub
train_data['level_num'] = train_data['level'].apply(level_to_num)
test_data['level_num'] = test_data['level'].apply(level_to_num)
train_data.drop('level', axis=1, inplace=True)
test_data.drop('level', axis=1, inplace=True)
numerical_fea.append('level_num')
category_fea.remove('level')


'\n# 5. 数值特征标准化\nscaler = StandardScaler()\ntrain_data[numerical_fea] = scaler.fit_transform(train_data[numerical_fea])\n'

In [172]:
# 对时间进行处理
import datetime
current_time = datetime.datetime.now()  # 或定义一个基准时间
original_time = ['issue_time', 'record_time', 'history_time']
for time_col in original_time:
    train_data[time_col + '_dt'] = pd.to_datetime(train_data[time_col], unit='s')  # Unix秒时间戳
    test_data[time_col + '_dt'] = pd.to_datetime(test_data[time_col], unit='s')
    train_data[time_col + '_age_days'] = (current_time - train_data[time_col + '_dt']).dt.days
    test_data[time_col + '_age_days'] = (current_time - test_data[time_col + '_dt']).dt.days
    numerical_fea.remove(time_col)
    numerical_fea.append(time_col + '_age_days')
    train_data.drop(time_col + '_dt',axis=1, inplace=True)
    test_data.drop(time_col + '_dt',axis=1, inplace=True)


train_data.drop(original_time,axis=1, inplace=True)
test_data.drop(original_time,axis=1, inplace=True)

#cat feature处理
catfeature_list = [col for col in train_data.columns if col != "label"]


In [173]:
def addStatementFeature(df, filepath):
    df_statement = pd.read_csv(filepath)
    
    # 方法1：保留 'id' 并排除 'label'
    cols_to_merge = [col for col in df_statement.columns if col != 'label']  # 保留 'id'
    
    # 方法2：显式添加 'id'（更安全）
    # cols_to_merge = ['id'] + [col for col in df_statement.columns if col not in ['id', 'label']]
    
    # 合并时确保右侧包含 'id'
    merged_df = pd.merge(
        df,
        df_statement[cols_to_merge],  # 此时包含 'id'
        on='id',
        how='left',
        suffixes=('', '_statement')
    )
    return merged_df

train_data = addStatementFeature(train_data,'train/train_statement_feature.csv')
test_data = addStatementFeature(test_data,'testaa/testaa_statement_feature.csv')
'''
catfeature_list.extend(['income_count', 'expense_count', 'big_income_count',
     'negative_balance_count'])
'''
for fea in catfeature_list:
    train_data[fea] = train_data[fea].astype('int64')
    test_data[fea] = test_data[fea].astype('int64')
 

In [175]:
category_fea

[]

In [176]:
numerical_fea

['id',
 'title',
 'career',
 'residence',
 'loan',
 'term',
 'interest_rate',
 'syndicated',
 'installment',
 'total_accounts',
 'balance_accounts',
 'balance_limit',
 'balance',
 'level_num',
 'issue_time_age_days',
 'record_time_age_days',
 'history_time_age_days']

In [177]:
catfeature_list

['id',
 'title',
 'career',
 'residence',
 'loan',
 'term',
 'interest_rate',
 'syndicated',
 'installment',
 'total_accounts',
 'balance_accounts',
 'balance_limit',
 'balance',
 'level_num',
 'issue_time_age_days',
 'record_time_age_days',
 'history_time_age_days']

In [178]:
train_data.head()

,id,title,career,residence,loan,term,interest_rate,syndicated,installment,total_accounts,...,income_expense_ratio,avg_income,income_std,expense_std,big_income_count,big_income_ratio,min_balance,final_balance,negative_balance_count,negative_balance_ratio
0,0,9,0,1,7200,36,10,0,1,17,...,0.202311,287.607143,269.295352,645.981279,0.0,0.000000,-48391.79,-47628.00,48.0,1.000000
1,1,8,10,0,21300,36,12,0,0,17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,8,7,1,10400,60,21,0,0,17,...,2.435264,360.993636,599.383992,86.884890,2.0,0.045455,-951.78,9361.34,4.0,0.083333
3,3,7,2,0,33050,36,16,0,1,17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,8,3,1,5200,36,14,0,0,17,...,1.515401,354.288506,556.915876,208.710804,5.0,0.057471,-4481.30,10483.20,39.0,0.419355


In [179]:
from sklearn.model_selection import train_test_split
# 1. 删除id列（正确用法）
train_data.drop('id', axis=1, inplace=True)  # 不赋值，直接修改原对象
catfeature_list.remove('id')
feature_list = [col for col in train_data.columns if col != "label"]
X_train, X_validation, y_train, y_validation = train_test_split(train_data.loc[:, feature_list], train_data.loc[:, 'label'], test_size=0.2 , random_state=2000)

In [180]:
X_train.head()

,title,career,residence,loan,term,interest_rate,syndicated,installment,total_accounts,balance_accounts,...,income_expense_ratio,avg_income,income_std,expense_std,big_income_count,big_income_ratio,min_balance,final_balance,negative_balance_count,negative_balance_ratio
12232,7,10,0,5200,36,9,0,1,41,15,...,0.773732,409.711587,697.243115,402.490795,3.0,0.047619,-7548.34,-7548.34,19.0,0.275362
12974,6,7,0,33050,36,19,0,0,15,8,...,0.438832,370.394231,366.761969,603.304386,0.0,0.000000,-25589.99,-24629.90,56.0,0.982456
47946,1,4,1,11000,12,30,0,0,10,6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6185,9,7,1,15350,36,13,0,0,43,17,...,0.838846,336.102632,479.624971,477.597515,3.0,0.039474,-11448.60,-4907.31,81.0,0.987805
29067,0,4,2,9000,12,10,0,0,5,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# 3. 模型建立与训练
# 确保 Notebook 内联显示图形
%matplotlib inline 
from catboost import CatBoostClassifier

model = CatBoostClassifier(iterations=2000,
                           #task_type='GPU',
                           #bootstrap_type='Poisson',
                           task_type='CPU',
                           cat_features=catfeature_list,
                           eval_metric='AUC',
                           logging_level='Verbose',
                           learning_rate=0.03,
                           depth=6, 
                           l2_leaf_reg=5,
                           loss_function='Logloss',
                            early_stopping_rounds=300,
                           scale_pos_weight=(len(y_train[y_train==0])/len(y_train[y_train == 1])),
                            random_seed= 42
                           )
model.fit(X_train.loc[:, feature_list], y_train, 
          eval_set=(X_validation.loc[:, feature_list], y_validation), plot=True)

plt.show()


MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

0:	test: 0.6258169	best: 0.6258169 (0)	total: 163ms	remaining: 5m 25s
1:	test: 0.6363709	best: 0.6363709 (1)	total: 272ms	remaining: 4m 31s
2:	test: 0.6396401	best: 0.6396401 (2)	total: 417ms	remaining: 4m 37s
3:	test: 0.6414251	best: 0.6414251 (3)	total: 517ms	remaining: 4m 18s
4:	test: 0.6404479	best: 0.6414251 (3)	total: 623ms	remaining: 4m 8s
5:	test: 0.6428357	best: 0.6428357 (5)	total: 706ms	remaining: 3m 54s
6:	test: 0.6427970	best: 0.6428357 (5)	total: 765ms	remaining: 3m 37s
7:	test: 0.6445256	best: 0.6445256 (7)	total: 839ms	remaining: 3m 29s
8:	test: 0.6447654	best: 0.6447654 (8)	total: 914ms	remaining: 3m 22s
9:	test: 0.6466274	best: 0.6466274 (9)	total: 995ms	remaining: 3m 17s
10:	test: 0.6466982	best: 0.6466982 (10)	total: 1.06s	remaining: 3m 12s
11:	test: 0.6459309	best: 0.6466982 (10)	total: 1.14s	remaining: 3m 8s
12:	test: 0.6463949	best: 0.6466982 (10)	total: 1.23s	remaining: 3m 7s
13:	test: 0.6464098	best: 0.6466982 (10)	total: 1.29s	remaining: 3m 3s
14:	test: 0.6465

In [ ]:
# 4.测试集预测
preds = model.predict_proba(test_data[feature_list])
preds


array([[0.74520002, 0.25479998],
       [0.88054788, 0.11945212],
       [0.59113652, 0.40886348],
       ...,
       [0.47910983, 0.52089017],
       [0.48435385, 0.51564615],
       [0.33375663, 0.66624337]])

In [187]:

test_data['label'] = preds[:, 1]  # 取正类的概率存储
test_data[['id', 'label']].to_csv('submission.csv', index=False)
